In [1]:
# Lab type: review
# Course: AI401 — AI Applications with LLMs
# Lesson: Orchestration Patterns: When to Use Frameworks and When Not To
# Task: Compare a LangChain implementation and a direct-API async implementation of the same batch classification task, then answer judgment questions

In [2]:
# Install the Anthropic library
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 15.9 MB/s eta 0:00:00


To use the Anthropic API, you'll need an API key. If you don't already have one, create a key on the [Anthropic console](https://console.anthropic.com/settings/keys).

In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `ANTHROPIC_API_KEY`. Then pass the key to the client initialization.

In [3]:
import os

try:
    # Attempt to import google.colab.userdata, which is only available in Colab
    from google.colab import userdata

    # Fetch the API key from Colab's secrets manager
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("Anthropic API key loaded from Colab secrets.")
except ImportError:
    # If not in Colab, try to load from a .env file using python-dotenv
    try:
        from dotenv import load_dotenv
        load_dotenv()
        ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
        if ANTHROPIC_API_KEY:
            print("Anthropic API key loaded from .env file.")
        else:
            print("Anthropic API key not found in .env file. Please ensure ANTHROPIC_API_KEY is set.")
    except ImportError:
        print("python-dotenv not installed. Please install it (`pip install python-dotenv`) or ensure ANTHROPIC_API_KEY is set as an environment variable.")
        ANTHROPIC_API_KEY = None

# Ensure the API key is not None before proceeding, or handle the error appropriately
if ANTHROPIC_API_KEY is None:
    raise ValueError("ANTHROPIC_API_KEY is not set. Please set it in Colab secrets or a .env file.")

Anthropic API key loaded from Colab secrets.


# Lab: Reviewing Orchestration Approaches

Two engineers have implemented the same batch ticket classification pipeline. Implementation A uses LangChain; Implementation B uses the Anthropic SDK directly with async/await and bounded concurrency.

Both produce correct classifications. Your task: read both implementations, run the inspection cells, and answer the judgment questions.

## Setup

In [4]:
import asyncio
import time
import anthropic

# Sample tickets for testing
TICKETS = [
    "Payment system down — all transactions failing",
    "Package not delivered, tracking shows it's still in transit",
    "App crashes immediately on login since yesterday's update",
    "Invoice shows incorrect amount, I was charged twice",
    "How do I update my billing address?",
    "Subscription auto-renewed but I cancelled last month",
    "Dashboard not loading for the last two hours",
    "Delivery estimated 3 days ago — nothing arrived",
]

CLASSIFY_SYSTEM = (
    "Classify the support ticket into exactly one of: billing, technical, shipping, other. "
    "Think through the ticket content, then state your classification "
    "on a new line in this exact format: CLASSIFICATION: <label>"
)

## Implementation A: LangChain

In [5]:
# Implementation A uses LangChain's ChatAnthropic + batch processing
# NOTE: This cell shows the structure — it will raise ImportError if
# langchain_anthropic is not installed, which is expected in this environment.
# You are reviewing the code, not running it.

IMPL_A_CODE = '''
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

def classify_batch_langchain(tickets: list[str]) -> list[str]:
    llm = ChatAnthropic(
        model="claude-haiku-4-5-20251001",
        max_tokens=128,
    )
    messages_batch = [
        [SystemMessage(content=CLASSIFY_SYSTEM), HumanMessage(content=t)]
        for t in tickets
    ]
    responses = llm.batch(messages_batch)
    results = []
    for r in responses:
        text = r.content
        import re
        match = re.search(r"CLASSIFICATION: (\\w+)", text)
        results.append(match.group(1).lower() if match else "unknown")
    return results
'''

print('Implementation A — LangChain batch:')
print(IMPL_A_CODE)

Implementation A — LangChain batch:

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

def classify_batch_langchain(tickets: list[str]) -> list[str]:
    llm = ChatAnthropic(
        model="claude-haiku-4-5-20251001",
        max_tokens=128,
    )
    messages_batch = [
        [SystemMessage(content=CLASSIFY_SYSTEM), HumanMessage(content=t)]
        for t in tickets
    ]
    responses = llm.batch(messages_batch)
    results = []
    for r in responses:
        text = r.content
        import re
        match = re.search(r"CLASSIFICATION: (\w+)", text)
        results.append(match.group(1).lower() if match else "unknown")
    return results



## Implementation B: Direct Anthropic SDK with async

In [6]:
import re

async_client = anthropic.AsyncAnthropic(api_key=ANTHROPIC_API_KEY)


async def classify_ticket(ticket: str, semaphore: asyncio.Semaphore) -> dict:
    """Classify a single ticket. Bounded by semaphore to avoid rate-limit bursts."""
    async with semaphore:
        t0 = time.perf_counter()
        response = await async_client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=128,
            system=CLASSIFY_SYSTEM,
            messages=[{'role': 'user', 'content': ticket}],
        )
        latency_ms = round((time.perf_counter() - t0) * 1000, 1)
        text = response.content[0].text
        match = re.search(r'CLASSIFICATION: (\w+)', text)
        return {
            'ticket': ticket[:50],
            'category': match.group(1).lower() if match else 'unknown',
            'input_tokens': response.usage.input_tokens,
            'output_tokens': response.usage.output_tokens,
            'latency_ms': latency_ms,
        }


async def classify_batch_direct(tickets: list[str], max_concurrent: int = 5) -> list[dict]:
    """Classify all tickets with bounded concurrency."""
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [classify_ticket(t, semaphore) for t in tickets]
    return await asyncio.gather(*tasks)

## Inspection: run Implementation B

In [8]:
import nest_asyncio
nest_asyncio.apply()

t0 = time.perf_counter()
results = asyncio.run(classify_batch_direct(TICKETS, max_concurrent=5))
wall_time = time.perf_counter() - t0

print(f'Classified {len(results)} tickets in {wall_time:.2f}s')
print()
for r in results:
    print(f"{r['category']:12s}  {r['latency_ms']:6.0f}ms  "
          f"in={r['input_tokens']:3d} out={r['output_tokens']:3d}  "
          f"{r['ticket']}")

Classified 8 tickets in 2.29s

technical       1341ms  in= 57 out= 42  Payment system down — all transactions failing
shipping        1127ms  in= 61 out= 41  Package not delivered, tracking shows it's still i
technical       1306ms  in= 59 out= 53  App crashes immediately on login since yesterday's
billing         1132ms  in= 59 out= 33  Invoice shows incorrect amount, I was charged twic
billing         1120ms  in= 58 out= 27  How do I update my billing address?
billing         1140ms  in= 60 out= 39  Subscription auto-renewed but I cancelled last mon
technical       1076ms  in= 58 out= 30  Dashboard not loading for the last two hours
shipping        1149ms  in= 61 out= 40  Delivery estimated 3 days ago — nothing arrived


In [9]:
# Compute per-call statistics
latencies = [r['latency_ms'] for r in results]
input_tokens = [r['input_tokens'] for r in results]
output_tokens = [r['output_tokens'] for r in results]

print(f'Latency  — p50: {sorted(latencies)[len(latencies)//2]:.0f}ms, '
      f'max: {max(latencies):.0f}ms')
print(f'Input tokens   — avg: {sum(input_tokens)/len(input_tokens):.0f}')
print(f'Output tokens  — avg: {sum(output_tokens)/len(output_tokens):.0f}')

Latency  — p50: 1140ms, max: 1341ms
Input tokens   — avg: 59
Output tokens  — avg: 38


## Judgment question 1

> Implementation A hides the token count per request inside LangChain's internals. At what processing scale does this opacity become a practical problem, and what specific information are you missing that would help you control costs?

In [10]:
# Your answer:
#
# Scale threshold:
#
# Missing information:
#

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Scale threshold:** Token-count opacity becomes a practical problem once you have hundreds of calls per day where cost anomalies, runaway prompts, or unexpected caching failures are possible. At that scale, a single poorly-constructed prompt that slips through code review can silently inflate costs by 10–100× before anyone notices.

**Missing information:** Per-request input and output token counts (needed to set token budgets and detect unusually large prompts), cached-vs-uncached token breakdown (needed to verify prefix caching is active and measure actual savings), and the exact model version called (needed to confirm no silent fallback to a more expensive model occurred).

</details>

## Judgment question 2

> The semaphore in `classify_batch_direct` is set to `max_concurrent=5`. The comments in Lesson 7 suggest: rate_limit_rpm / (60 / avg_latency_seconds).

> Using the latency data from your run above, what `max_concurrent` would be appropriate for a 60 RPM rate limit tier?

In [11]:
# Calculate the appropriate max_concurrent for a 60 RPM limit
avg_latency_s = sum(latencies) / len(latencies) / 1000
rpm_limit = 60

# Your calculation:
# max_concurrent = rpm_limit * avg_latency_s / 60
max_concurrent_60rpm = rpm_limit * avg_latency_s / 60
print(f'Average latency : {avg_latency_s:.3f}s')
print(f'Max concurrent  : {max_concurrent_60rpm:.2f} → use {max(1, int(max_concurrent_60rpm))}')

Average latency : 1.174s
Max concurrent  : 1.17 → use 1


<details>
<summary>🔑 Reveal answer — Q2</summary>

**Formula:** `max_concurrent = rpm_limit × avg_latency_s / 60`

This keeps the number of in-flight requests low enough that they complete before the per-minute quota resets. For a 60 RPM limit with average latency of ~0.5 s (typical for Haiku on short classification tasks): `60 × 0.5 / 60 = 0.5` → round up to **1**. At 1 s average latency: `60 × 1.0 / 60 = 1` → still 1. The formula scales correctly: at 2 s average latency and 60 RPM you'd use 2, meaning at most 2 requests outstanding at once to stay under rate limits with comfortable headroom.

</details>

## Judgment question 3

> Implementation B logs `latency_ms` per call. What 3 additional fields would you add to the per-call result dict to make it possible to diagnose a performance regression in production?

In [12]:
# Your answer — list 3 fields with types and the diagnostic question each answers:
#
# Field 1:
#
# Field 2:
#
# Field 3:
#

<details>
<summary>🔑 Reveal answer — Q3</summary>

**Field 1 — `error: str | None`:** Captures the exception message when a call fails (API error, timeout, rate-limit). Without it, a systematic failure pattern (e.g., all tickets of a certain length timing out) is invisible until you notice the missing results.

**Field 2 — `request_id: str`:** The Anthropic `x-request-id` response header. When a specific call produces a wrong or unexpected output, the request ID lets you look it up in provider logs or open a support ticket with a precise reference — otherwise the call is unidentifiable.

**Field 3 — `cache_read_input_tokens: int`:** How many input tokens were served from the provider-side prompt cache (from the Anthropic usage object). Confirms that prefix caching is actually activating and lets you measure cost savings per call. If this is always 0, you know caching is misconfigured.

</details>

## Judgment question 4

> The lesson's framework adoption criteria are:
> - The pattern is well-understood and well-tested in the framework
> - You don't need visibility into the details the framework hides
> - The team has read and understands the framework source

> Name one scenario in this classification pipeline where you would choose Implementation A (LangChain) over Implementation B, and justify it against the three criteria.

In [13]:
# Your answer:
#
# Scenario:
#
# Justification against each criterion:
#  1.
#  2.
#  3.
#

<details>
<summary>🔑 Reveal answer — Q4</summary>

**Scenario:** Early in a project when you are prototyping across multiple LLM providers (Anthropic, OpenAI, and Bedrock) and need to switch providers with minimal code changes for A/B cost comparisons.

**Justification against the three criteria:**
1. *Well-understood and tested in the framework:* LangChain's provider-abstraction layer for `ChatAnthropic`, `ChatOpenAI`, and `BedrockChat` is one of the most heavily used and tested parts of the library — provider switching via a one-line constructor change is a documented, stable pattern.
2. *You don't need visibility into what the framework hides:* In a prototype phase, per-call token counts and latency breakdowns are not yet actionable — you're choosing a provider, not optimising a production budget.
3. *The team has read the framework source:* The relevant LangChain code for provider switching is a thin wrapper around the native SDK — small enough that a 30-minute code review covers it before committing to its use.

</details>